# Q4: Feature Engineering

**Phase 5:** Feature Engineering & Aggregation  
**Points: 9 points**

**Focus:** Create derived features, perform time-based aggregations, calculate rolling windows.

**Lecture Reference:** Lecture 11, Notebook 2 ([`11/demo/02_wrangling_feature_engineering.ipynb`](https://github.com/christopherseaman/datasci_217/blob/main/11/demo/02_wrangling_feature_engineering.ipynb)), Phase 5. Also see Lecture 09 (rolling windows).

---

## Setup

In [4]:
# Import libraries
import pandas as pd
import numpy as np
import os

# Load wrangled data from Q3
df = pd.read_csv('output/q3_wrangled_data.csv', parse_dates=['Measurement Timestamp'], index_col='Measurement Timestamp')
# Or if you saved without index:
# df = pd.read_csv('output/q3_wrangled_data.csv')
# df['Measurement Timestamp'] = pd.to_datetime(df['Measurement Timestamp'])
# df = df.set_index('Measurement Timestamp')
print(f"Loaded {len(df):,} records with datetime index")

Loaded 196,571 records with datetime index


---

## Objective

Create derived features, perform time-based aggregations, and calculate rolling windows for time series analysis.

**Time Series Note:** Rolling windows are essential for time series data. They capture temporal dependencies (e.g., 7-hour rolling mean captures short-term patterns). See **Lecture 09** for time series rolling window operations. For hourly data, common window sizes are 7-24 hours (capturing daily patterns). Use pandas `rolling()` method with `window` parameter to specify the number of periods.

---

## Required Artifacts

You must create exactly these 3 files in the `output/` directory:

### 1. `output/q4_features.csv`
**Format:** CSV file
**Content:** Dataset with all derived features added
**Requirements:**
- All original columns from Q3
- All new derived features added as columns
- **No index column** (save with `index=False`)

### 2. `output/q4_rolling_features.csv`
**Format:** CSV file
**Content:** Dataset with rolling window features
**Required Columns:**
- Original datetime column
- At least one rolling window calculation column (e.g., `water_temp_rolling_7h`, `air_temp_rolling_24h`)

**Requirements:**
- Must include at least one rolling window calculation
- Rolling window names should be descriptive (e.g., `temp_rolling_7h` for 7-hour rolling mean)
- **No index column** (save with `index=False`)

**Example columns:**
```csv
Measurement Timestamp,wind_speed_rolling_7h,humidity_rolling_24h,pressure_rolling_7h
2022-01-01 00:00:00,6.8,65.2,1013.5
2022-01-01 01:00:00,6.9,65.3,1013.6
...
```

**Note:** The example shows rolling windows of predictor variables (wind speed, humidity, pressure), not the target variable. If you're predicting Air Temperature, do NOT create rolling windows of Air Temperature - this causes data leakage.

### 3. `output/q4_feature_list.txt`
**Format:** Plain text file
**Content:** List of new features created (one per line)
**Requirements:**
- One feature name per line
- No extra text, just feature names
- Include all derived features, rolling features, and categorical features created

**Example format:**
```
temp_difference
temp_ratio
wind_speed_squared
comfort_index
water_temp_rolling_7h
air_temp_rolling_24h
wind_speed_rolling_7h
temp_category
wind_category
```

---

## Requirements Checklist

- [ ] Derived features created (differences, ratios, interactions, etc.)
- [ ] Time-based aggregations performed (by hour, day, month, etc.) - optional but recommended
- [ ] At least one rolling window calculation (rolling mean, rolling median, etc.)
- [ ] Categorical features created (if applicable)
- [ ] Feature list documented
- [ ] All 3 required artifacts saved with exact filenames

---

## Your Approach

1. **Create derived features** - Differences, ratios, interactions between variables (watch for division by zero)
2. **Calculate rolling windows** - Use `.rolling()` on predictor variables to capture temporal patterns

   ⚠️ **Data Leakage Warning:** Do not create ANY features that use your target variable - this includes rolling windows, differences, ratios, or interactions involving the target. For example, if predicting Air Temperature, do not create `air_temp * humidity` or `air_temp - wet_bulb`. Only derive features from other predictor variables.

3. **Create categorical features** - Bin continuous variables if useful (optional)
4. **Check for infinity values** - Ratios can produce infinity; replace with NaN and handle appropriately
5. **Document and save** - Remember to `reset_index()` before saving CSVs

---

## Decision Points

- **Derived features:** What relationships might be useful? Temperature differences? Ratios? Interactions between variables?
- **Rolling windows:** What window size makes sense? 7 hours? 24 hours? Consider the temporal scale of your data. For hourly data, 7-24 hours captures daily patterns.
- **Time-based aggregations:** Aggregate by hour? Day? Week? What temporal granularity is useful for your analysis?

---

## Checkpoint

After Q4, you should have:
- [ ] Derived features created
- [ ] At least one rolling window calculation
- [ ] Feature list documented
- [ ] All 3 artifacts saved: `q4_features.csv`, `q4_rolling_features.csv`, `q4_feature_list.txt`

---

**Next:** Continue to `q5_pattern_analysis.md` for Pattern Analysis.


In [7]:
# Question 1. Features csv 
print("="*80)
print("Q4.1 Derived Features")
print("="*80)

# Load wrangled data from Q3
print("\n Load wrangled data from Q3")
print("-" * 80)
df = pd.read_csv('output/q3_wrangled_data.csv', 
                 parse_dates=['Measurement Timestamp'], 
                 index_col='Measurement Timestamp')
df = df.sort_index()

print(f"  Loaded: output/q3_wrangled_data.csv")
print(f"  Records: {len(df):,}")
print(f"  Date range: {df.index.min()} to {df.index.max()}")
print(f"  Original columns: {list(df.columns)}")
print(f"  Datetime index: {df.index.name}")

# Identify numeric columns for feature analysis
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumeric columns for feature analysis ({len(numeric_cols)}):")
for col in numeric_cols:
    print(f"  - {col}")

# Store original column count
original_col_count = len(df.columns)

print("\n" + "="*80)
print("SECTION 1: TEMPORAL FEATURES")
print("="*80)

print("\n EXTRACTING TEMPORAL FEATURES")
print("-" * 80)
print("Keeping only the most informative temporal features")

# Essential temporal features only
df['hour'] = df.index.hour
df['day_of_week'] = df.index.dayofweek
df['month'] = df.index.month
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['is_business_hours'] = ((df['hour'] >= 9) & (df['hour'] <= 17)).astype(int)

temporal_features = ['hour', 'day_of_week', 'month', 'is_weekend', 'is_business_hours']

print(f"  Created {len(temporal_features)} temporal features:")
for feat in temporal_features:
    print(f"  - {feat}")

# ROLLING WINDOW FEATURES - MEAN & STD ONLY
print("\n" + "="*80)
print("SECTION 2: ROLLING WINDOW FEATURES (Weekly)")
print("="*80)

print("\n CALCULATING ROLLING WINDOWS (168 hours = 1 week)")
print("-" * 80)
print("Keep: mean (central tendency) and std (variability)")
print("Exclude: range (redundant with std)")

rolling_features = []
rolling_dfs = []

# For each numeric column, create rolling window features
for col in numeric_cols:
    print(f"\nProcessing: {col}")
    
    # Create dictionary to store rolling features for this column
    rolling_dict = {}
    
    # 168-hour (1 week) rolling windows - mean and std only
    rolling_dict[f'{col}_rolling_mean_168h'] = df[col].rolling(window=168, min_periods=1).mean()
    rolling_dict[f'{col}_rolling_std_168h'] = df[col].rolling(window=168, min_periods=1).std()
    
    # Convert to DataFrame and add to list
    rolling_df = pd.DataFrame(rolling_dict, index=df.index)
    rolling_dfs.append(rolling_df)
    
    window_features = list(rolling_dict.keys())
    rolling_features.extend(window_features)
    
    print(f"   Created 2 rolling window features (mean, std)")

# Concatenate all rolling features at once
print(f"\n Concatenating all rolling window features...")
rolling_features_df = pd.concat(rolling_dfs, axis=1)
df = pd.concat([df, rolling_features_df], axis=1)

print(f" Total rolling window features created: {len(rolling_features)}")

# LAG FEATURES - 24H ONLY
print("\n" + "="*80)
print("SECTION 3: LAG FEATURES (24-hour only)")
print("="*80)

print("\n CREATING LAG FEATURES")
print("-" * 80)
print("Rationale: 24h lag captures 'same time yesterday' pattern")

lag_features = []
lag_dfs = []

for col in numeric_cols:
    lag_dict = {}
    
    # 24-hour lag and change only
    lag_dict[f'{col}_lag_24h'] = df[col].shift(24)
    lag_dict[f'{col}_change_24h'] = df[col] - lag_dict[f'{col}_lag_24h']
    
    # Convert to DataFrame
    lag_df = pd.DataFrame(lag_dict, index=df.index)
    lag_dfs.append(lag_df)
    
    lag_cols = list(lag_dict.keys())
    lag_features.extend(lag_cols)

# Concatenate all lag features
print(f"\n  Concatenating all lag features...")
lag_features_df = pd.concat(lag_dfs, axis=1)
df = pd.concat([df, lag_features_df], axis=1)

print(f"  Created {len(lag_features)} lag features:")
print(f"  - 24h lag (same time yesterday)")
print(f"  - 24h change (difference from yesterday)")

# TIME-BASED AGGREGATIONS - HOURLY AVERAGE ONLY
print("\n" + "="*80)
print("SECTION 4: TIME-BASED AGGREGATIONS (Hourly patterns)")
print("="*80)

print("\n CALCULATING HOURLY AGGREGATIONS")
print("-" * 80)
print("Include: hourly_avg (typical value by hour)")

aggregation_features = []
agg_dfs = []

for col in numeric_cols:
    agg_dict = {}
    
    # Hourly mean across all data (typical value for this hour of day)
    hourly_mean = df.groupby(df.index.hour)[col].transform('mean')
    agg_dict[f'{col}_hourly_avg'] = hourly_mean
    
    # Convert to DataFrame
    agg_df = pd.DataFrame(agg_dict, index=df.index)
    agg_dfs.append(agg_df)
    
    agg_cols = list(agg_dict.keys())
    aggregation_features.extend(agg_cols)

# Concatenate all aggregation features
print(f"\n  Concatenating all aggregation features...")
agg_features_df = pd.concat(agg_dfs, axis=1)
df = pd.concat([df, agg_features_df], axis=1)

print(f"  Created {len(aggregation_features)} aggregation features:")
print(f"  - Hourly averages (typical value by hour of day)")

# SUMMARY OF ALL FEATURES CREATED
print("\n" + "="*80)
print("FEATURE ENGINEERING SUMMARY")
print("="*80)

new_col_count = len(df.columns)
derived_features_count = new_col_count - original_col_count

print(f"\nOriginal columns: {original_col_count}")
print(f"New derived features: {derived_features_count}")
print(f"Total columns: {new_col_count}")

print(f"\nBreakdown:")
print(f"  - Temporal features: {len(temporal_features)}")
print(f"  - Rolling window features (168h): {len(rolling_features)}")
print(f"  - Lag features (24h): {len(lag_features)}")
print(f"  - Aggregation features (hourly): {len(aggregation_features)}")
print(f"  TOTAL: {len(temporal_features) + len(rolling_features) + len(lag_features) + len(aggregation_features)}")

print(f"\nFeatures per numeric column: {(len(rolling_features) + len(lag_features) + len(aggregation_features)) // len(numeric_cols)}")
print(f"  (2 rolling + 2 lag + 1 aggregation = 5 features per column)")

# Save Derived Features List
print("\n" + "="*80)
print("SAVING FEATURES")
print("="*80)

print("\n SAVING DATASET WITH DERIVED FEATURES")
print("-" * 80)

# Reset index to save datetime as column
df_to_save = df.reset_index()

print(f"Columns to save: {len(df_to_save.columns)}")
print(f"  - Datetime column: Measurement Timestamp")
print(f"  - Original data columns: {original_col_count}")
print(f"  - Derived feature columns: {derived_features_count}")

# Save to CSV
df_to_save.to_csv('output/q4_features.csv', index=False)

print(f"\n  Saved to: output/q4_features.csv")
print(f"  Shape: {df_to_save.shape[0]:,} rows × {df_to_save.shape[1]} columns")
print(f"  Index parameter: index=False")

print("\n" + "="*80)
print("FEATURE ENGINEERING COMPLETE")
print("="*80)


Q4.1 Derived Features

 Load wrangled data from Q3
--------------------------------------------------------------------------------
  Loaded: output/q3_wrangled_data.csv
  Records: 196,571
  Date range: 2015-04-25 09:00:00 to 2025-12-09 00:00:00
  Original columns: ['Station Name', 'Air Temperature', 'Wet Bulb Temperature', 'Humidity', 'Rain Intensity', 'Interval Rain', 'Total Rain', 'Precipitation Type', 'Wind Direction', 'Wind Speed', 'Maximum Wind Speed', 'Barometric Pressure', 'Solar Radiation', 'Heading', 'Battery Life', 'Measurement Timestamp Label', 'Measurement ID']
  Datetime index: Measurement Timestamp

Numeric columns for feature analysis (14):
  - Air Temperature
  - Wet Bulb Temperature
  - Humidity
  - Rain Intensity
  - Interval Rain
  - Total Rain
  - Precipitation Type
  - Wind Direction
  - Wind Speed
  - Maximum Wind Speed
  - Barometric Pressure
  - Solar Radiation
  - Heading
  - Battery Life

SECTION 1: TEMPORAL FEATURES

 EXTRACTING TEMPORAL FEATURES
-----------

In [8]:
# 2. Rolling Window Features csv
print("="*80)
print("Q4.2: EXTRACTING ROLLING WINDOW FEATURES")
print("="*80)

# Load the full features dataset
print("\n LOADING FULL FEATURES DATASET")
print("-" * 80)
df = pd.read_csv('output/q4_features.csv', 
                 parse_dates=['Measurement Timestamp'])

print(f"  Loaded: output/q4_features.csv")
print(f"  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Identify rolling window feature columns
print("\n IDENTIFYING ROLLING WINDOW FEATURES")
print("-" * 80)

rolling_cols = [col for col in df.columns if 'rolling' in col.lower()]

print(f"  Found {len(rolling_cols)} rolling window feature columns")

# Group by window type
rolling_168h = [col for col in rolling_cols if '168h' in col]

print(f"\nBreakdown:")
print(f"  - 168-hour (1 week) rolling features: {len(rolling_168h)}")

# Verify we only have the expected features (mean and std)
rolling_mean = [col for col in rolling_168h if '_mean_' in col]
rolling_std = [col for col in rolling_168h if '_std_' in col]

print(f"\nFeature types:")
print(f"  - Rolling mean (168h): {len(rolling_mean)}")
print(f"  - Rolling std (168h): {len(rolling_std)}")

# Show all rolling feature names
print(f"\nRolling feature names ({len(rolling_cols)} total):")
for col in rolling_cols:
    print(f"  - {col}")


# Create dataset with datetime and rolling features
print("\n CREATING ROLLING FEATURES DATASET")
print("-" * 80)

# Select datetime column and all rolling features
rolling_features_cols = ['Measurement Timestamp'] + rolling_cols
df_rolling = df[rolling_features_cols].copy()

print(f" Created rolling features dataset")
print(f"  Columns: {len(df_rolling.columns)}")
print(f"  - Datetime column: 1")
print(f"  - Rolling feature columns: {len(rolling_cols)}")

# Show statistics for rolling features
print("\n4. ROLLING FEATURES STATISTICS")
print("-" * 80)

# Check for missing values
missing_counts = df_rolling[rolling_cols].isnull().sum()
features_with_missing = missing_counts[missing_counts > 0]

if len(features_with_missing) > 0:
    print(f" ! Features with missing values: {len(features_with_missing)}")
    print(f"Note: Missing values at start of time series are expected")
    print(f"\nTop features with missing values:")
    for feat, count in features_with_missing.head(5).items():
        pct = (count / len(df_rolling)) * 100
        print(f"  - {feat}: {count:,} ({pct:.2f}%)")
else:
    print("  No missing values in rolling features")

# Sample data preview
print("\n5. SAMPLE DATA PREVIEW")
print("-" * 80)
print("\nFirst 5 rows (datetime + first 5 rolling features):")
preview_cols = ['Measurement Timestamp'] + rolling_cols[:5]
print(df_rolling[preview_cols].head())

# Save rolling features dataset
print("\n SAVING ROLLING FEATURES DATASET")
print("-" * 80)

df_rolling.to_csv('output/q4_rolling_features.csv', index=False)

print(f"  Saved to: output/q4_rolling_features.csv")
print(f"  Shape: {df_rolling.shape[0]:,} rows × {df_rolling.shape[1]} columns")
print(f"  Index parameter: index=False")

# Summary
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f" Extracted {len(rolling_cols)} rolling window features")
print(f" Dataset contains:")
print(f"  - 1 datetime column (Measurement Timestamp)")
print(f"  - {len(rolling_168h)} 168-hour (weekly) rolling features")
print(f"    - {len(rolling_mean)} rolling means")
print(f"    - {len(rolling_std)} rolling standard deviations")
print(f" Total rows: {df_rolling.shape[0]:,}")
print(f" Saved to: output/q4_rolling_features.csv")

print("\n" + "="*80)
print("ROLLING FEATURES EXTRACTION COMPLETE")
print("="*80)

Q4.2: EXTRACTING ROLLING WINDOW FEATURES

 LOADING FULL FEATURES DATASET
--------------------------------------------------------------------------------
  Loaded: output/q4_features.csv
  Shape: 196,571 rows × 93 columns

 IDENTIFYING ROLLING WINDOW FEATURES
--------------------------------------------------------------------------------
  Found 28 rolling window feature columns

Breakdown:
  - 168-hour (1 week) rolling features: 28

Feature types:
  - Rolling mean (168h): 14
  - Rolling std (168h): 14

Rolling feature names (28 total):
  - Air Temperature_rolling_mean_168h
  - Air Temperature_rolling_std_168h
  - Wet Bulb Temperature_rolling_mean_168h
  - Wet Bulb Temperature_rolling_std_168h
  - Humidity_rolling_mean_168h
  - Humidity_rolling_std_168h
  - Rain Intensity_rolling_mean_168h
  - Rain Intensity_rolling_std_168h
  - Interval Rain_rolling_mean_168h
  - Interval Rain_rolling_std_168h
  - Total Rain_rolling_mean_168h
  - Total Rain_rolling_std_168h
  - Precipitation Type_rol

In [9]:
# 4.3 New Features txt 
print("="*80)
print("Q4.3: GENERATING FEATURE LIST")
print("="*80)

# Load original wrangled data to identify original columns
print("\n LOADING ORIGINAL AND FEATURES DATASETS")
print("-" * 80)

df_original = pd.read_csv('output/q3_wrangled_data.csv')
original_cols = set(df_original.columns)

print(f"  Loaded original data: output/q3_wrangled_data.csv")
print(f"  Original columns: {len(original_cols)}")

df_features = pd.read_csv('output/q4_features.csv')
all_cols = set(df_features.columns)

print(f"  Loaded features data: output/q4_features.csv")
print(f"  Total columns: {len(all_cols)}")

# Identify new features (columns in features dataset but not in original)
print("\n IDENTIFYING NEW FEATURES")
print("-" * 80)

new_features = sorted(all_cols - original_cols)

print(f" Found {len(new_features)} new features")

# Categorize features by type
print("\n CATEGORIZING FEATURES")
print("-" * 80)

feature_categories = {
    'temporal': [],
    'rolling_168h': [],
    'lag_24h': [],
    'change_24h': [],
    'aggregation_hourly': [],
    'other': []
}

for feat in new_features:
    feat_lower = feat.lower()
    
    # Temporal features
    if any(x in feat_lower for x in ['hour', 'day_of_week', 'month', 'weekend', 'business']):
        feature_categories['temporal'].append(feat)
    # Rolling window features (168h only)
    elif 'rolling' in feat_lower and '168h' in feat_lower:
        feature_categories['rolling_168h'].append(feat)
    # Lag features (24h only)
    elif 'lag_24h' in feat_lower:
        feature_categories['lag_24h'].append(feat)
    # Change features (24h only)
    elif 'change_24h' in feat_lower:
        feature_categories['change_24h'].append(feat)
    # Aggregation features (hourly average only)
    elif 'hourly_avg' in feat_lower:
        feature_categories['aggregation_hourly'].append(feat)
    else:
        feature_categories['other'].append(feat)

# Display category breakdown with counts
print("\nFeature breakdown by category:")
total_categorized = 0
for category, features in feature_categories.items():
    if features:
        count = len(features)
        total_categorized += count
        print(f"  - {category}: {count} features")

print(f"\nTotal categorized: {total_categorized}")
if feature_categories['other']:
    print(f"  Warning: {len(feature_categories['other'])} features in 'other' category")


# Generate report content
print("\n GENERATING DETAILED REPORT")
print("-" * 80)

report_lines = []
report_lines.append("="*80)
report_lines.append("FEATURE ENGINEERING REPORT")
report_lines.append("="*80)
report_lines.append("")
report_lines.append(f"Total new features created: {len(new_features)}")
report_lines.append(f"Original columns: {len(original_cols)}")
report_lines.append(f"Total columns after feature engineering: {len(all_cols)}")
report_lines.append("")
report_lines.append("="*80)
report_lines.append("FEATURE CATEGORIES")
report_lines.append("="*80)
report_lines.append("")

# Temporal Features
if feature_categories['temporal']:
    report_lines.append(f"1. TEMPORAL FEATURES ({len(feature_categories['temporal'])})")
    report_lines.append("-" * 80)
    report_lines.append("Description: Time-based features capturing cyclical patterns")
    report_lines.append("")
    for feat in sorted(feature_categories['temporal']):
        report_lines.append(f"  - {feat}")
    report_lines.append("")

# Rolling Window Features (168h)
if feature_categories['rolling_168h']:
    report_lines.append(f"2. ROLLING WINDOW FEATURES - 168h/Weekly ({len(feature_categories['rolling_168h'])})")
    report_lines.append("-" * 80)
    report_lines.append("Description: Statistical measures over 1-week rolling windows")
    report_lines.append("Window size: 168 hours (1 week)")
    report_lines.append("Captures: Weekday/weekend patterns in city data")
    report_lines.append("")
    
    # Group by feature type (mean, std)
    rolling_mean = sorted([f for f in feature_categories['rolling_168h'] if '_mean_' in f])
    rolling_std = sorted([f for f in feature_categories['rolling_168h'] if '_std_' in f])
    
    if rolling_mean:
        report_lines.append(f"  Rolling Mean ({len(rolling_mean)}):")
        for feat in rolling_mean:
            report_lines.append(f"    - {feat}")
        report_lines.append("")
    
    if rolling_std:
        report_lines.append(f"  Rolling Standard Deviation ({len(rolling_std)}):")
        for feat in rolling_std:
            report_lines.append(f"    - {feat}")
        report_lines.append("")

# Lag Features (24h)
if feature_categories['lag_24h']:
    report_lines.append(f"3. LAG FEATURES - 24h ({len(feature_categories['lag_24h'])})")
    report_lines.append("-" * 80)
    report_lines.append("Description: Previous values from 24 hours ago")
    report_lines.append("Purpose: Captures 'same time yesterday' pattern")
    report_lines.append("")
    for feat in sorted(feature_categories['lag_24h']):
        report_lines.append(f"  - {feat}")
    report_lines.append("")

# Change Features (24h)
if feature_categories['change_24h']:
    report_lines.append(f"4. CHANGE FEATURES - 24h ({len(feature_categories['change_24h'])})")
    report_lines.append("-" * 80)
    report_lines.append("Description: Difference from 24 hours ago")
    report_lines.append("Purpose: Captures day-over-day changes")
    report_lines.append("")
    for feat in sorted(feature_categories['change_24h']):
        report_lines.append(f"  - {feat}")
    report_lines.append("")

# Aggregation Features (hourly)
if feature_categories['aggregation_hourly']:
    report_lines.append(f"5. HOURLY AGGREGATION FEATURES ({len(feature_categories['aggregation_hourly'])})")
    report_lines.append("-" * 80)
    report_lines.append("Description: Average value by hour of day across all data")
    report_lines.append("Purpose: Captures typical hourly patterns (e.g., rush hour effects)")
    report_lines.append("")
    for feat in sorted(feature_categories['aggregation_hourly']):
        report_lines.append(f"  - {feat}")
    report_lines.append("")

# Other features (if any)
if feature_categories['other']:
    report_lines.append(f"6. OTHER FEATURES ({len(feature_categories['other'])})")
    report_lines.append("-" * 80)
    for feat in sorted(feature_categories['other']):
        report_lines.append(f"  - {feat}")
    report_lines.append("")

# Save detailed report
print("\n SAVING FEATURE REPORTS")
print("-" * 80)

# Save simple list
with open('output/q4_feature_list.txt', 'w') as f:
    for feature in sorted(new_features):
        f.write(feature + '\n')

print(f"  Saved list to: output/q4_feature_list.txt")
print(f"  Format: One feature name per line")

# Display summary
print("\n FEATURE SUMMARY")
print("-" * 80)
print(f"Total new features: {len(new_features)}")
print("\nBy category:")
for category, features in feature_categories.items():
    if features:
        print(f"  {category}: {len(features)}")

print("\n" + "="*80)
print("FEATURE LIST GENERATION COMPLETE")
print("="*80)
print("\nGenerated files:")
print("  1. output/q4_feature_list.txt (simple list)")
print("="*80)

Q4.3: GENERATING FEATURE LIST

 LOADING ORIGINAL AND FEATURES DATASETS
--------------------------------------------------------------------------------
  Loaded original data: output/q3_wrangled_data.csv
  Original columns: 18
  Loaded features data: output/q4_features.csv
  Total columns: 93

 IDENTIFYING NEW FEATURES
--------------------------------------------------------------------------------
 Found 75 new features

 CATEGORIZING FEATURES
--------------------------------------------------------------------------------

Feature breakdown by category:
  - temporal: 19 features
  - rolling_168h: 28 features
  - lag_24h: 14 features
  - change_24h: 14 features

Total categorized: 75

 GENERATING DETAILED REPORT
--------------------------------------------------------------------------------

 SAVING FEATURE REPORTS
--------------------------------------------------------------------------------
  Saved list to: output/q4_feature_list.txt
  Format: One feature name per line

 FEATURE 